1 - create a file level isntru test review
2 - aggregate the results to create a repo level instru review

In [8]:
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple, Dict, Set

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# Optional: allow unreferenced scripts in known CI paths (kept False for minimum false-positives)
LOOSEN_KNOWN_CI_PATHS = False
KNOWN_CI_DIRS = ('.github/scripts', '.github/workflows/scripts', 'ci', 'scripts/ci', 'tools/ci')

STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# --- Regex helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M | re.S) -> List[Pattern]:
    # DOTALL so multi-line Gradle blocks match (managedDevices {...})
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def collect_matches(named_patterns: List[Tuple[str, List[Pattern]]], text: str) -> List[str]:
    hits = []
    for label, pats in named_patterns:
        if any_match(pats, text):
            hits.append(label)
    return hits

# --- Comment scrubbing ---
BLOCK_COMMENT_RE = re.compile(r'/\*.*?\*/', re.S)            # Gradle/KTS block
LINE_COMMENT_RE = re.compile(r'(?m)^\s*//.*?$')              # Gradle/KTS line
YAML_SHELL_LINE_COMMENT_RE = re.compile(r'(?m)^\s*#.*?$')    # YAML/SH/PS1 line
BAT_REM_LINE_RE = re.compile(r'(?mi)^\s*(REM\b.*)$')         # BAT/CMD
BAT_COLON_COMMENT_RE = re.compile(r'(?m)^\s*::.*$')          # BAT/CMD
PS1_BLOCK_COMMENT_RE = re.compile(r'<#.*?#>', re.S | re.I)   # PS1

def strip_comments(raw: str, ext: str) -> str:
    e = ext.lower()
    txt = raw
    if e in ('.yml', '.yaml', '.sh'):
        return YAML_SHELL_LINE_COMMENT_RE.sub('', txt)
    if e in ('.gradle', '.kts'):
        txt = BLOCK_COMMENT_RE.sub('', txt)
        txt = LINE_COMMENT_RE.sub('', txt)
        return txt
    if e in ('.bat', '.cmd'):
        txt = BAT_REM_LINE_RE.sub('', txt)
        txt = BAT_COLON_COMMENT_RE.sub('', txt)
        return txt
    if e == '.ps1':
        txt = PS1_BLOCK_COMMENT_RE.sub('', txt)
        txt = YAML_SHELL_LINE_COMMENT_RE.sub('', txt)
        return txt
    return txt  # .json and others

# === DEVICE SETUP SOURCES ===
# 1) Real-device evidence (explicit non-emulator serial or SaaS real-device labs).
REAL_DEVICE_SOURCES = [
    ('adb -s',                 [r'(?m)^(?!\s*#)\s*adb\s+-s\s+(?P<serial>\S+)\b']),
    ('appcenter test android', [r'(?m)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
    ('saucectl',               [r'(?m)^(?!\s*#)\s*saucectl(\s+run)?\b']),
    ('browserstack/bstack',    [r'(?m)^(?!\s*#)\s*(browserstack|bstack)\b']),
]

# 2) Emulator evidence (manual or action-based).
EMULATOR_SOURCES = [
    ('sdkmanager/avdmanager',                 [r'(?m)^(?!\s*#)\s*(sdkmanager|avdmanager)\b']),
    ('emulator -avd/@',                       [r'(?m)^(?!\s*#)\s*emulator\s+(-avd|@)\S+']),
    ('android-wait-for-emulator',             [r'(?m)^(?!\s*#)\s*android-wait-for-emulator\b']),
    ('start-emulator.sh',                     [r'(?m)^(?!\s*#)\s*start-emulator\.sh\b']),
    ('reactivecircus/android-emulator-runner',[r'uses:\s*reactivecircus/android-emulator-runner']),
    ('actions/setup-android',                 [r'uses:\s*actions/setup-android']),
    ('pierotofy/setup-android',               [r'uses:\s*pierotofy/setup-android']),
    ('api-level key',                         [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ('ABI/arch key',                          [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ('target image',                          [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ('device name',                           [r'\b(avd[-_ ]?name|device|profile)\b\s*:\s*(pixel|nexus|android)']),
]

# 3) Third-party lab (Firebase Test Lab, etc.).
THIRD_PARTY_SOURCES = [
    ('gcloud firebase',                       [r'(?m)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ('test_matrix.json/firebase.json',        [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

# 4) Generic ADB (no type implied by itself).
ADB_GENERIC_SOURCES = [
    ('adb devices',           [r'(?m)^(?!\s*#)\s*adb\s+devices\b']),
    ('adb get-state',         [r'(?m)^(?!\s*#)\s*adb\s+get-state\b']),
    ('adb get-serialno',      [r'(?m)^(?!\s*#)\s*adb\s+get-serialno\b']),
    ('adb shell',             [r'(?m)^(?!\s*#)\s*adb\s+shell\b']),
    ('adb input',             [r'(?m)^(?!\s*#)\s*adb\s+shell\s+input\b']),
    ('adb settings',          [r'(?m)^(?!\s*#)\s*adb\s+shell\s+settings\b']),
    ('adb pm grant',          [r'(?m)^(?!\s*#)\s*adb\s+shell\s+pm\s+grant\b']),
    ('adb install (no -s)',   [r'(?m)^(?!\s*#)\s*adb\s+install(\s+-r)?\b(?!.*\s-s\s)']),
]

# 5) ADB hints that imply a type (promotion rules).
ADB_HINT_SOURCES = [
    ('adb -e (emulator)',     [r'(?m)^(?!\s*#)\s*adb\s+-e\b']),
    ('adb -d (real)',         [r'(?m)^(?!\s*#)\s*adb\s+-d\b']),
    ('adb -s emulator-XXXX',  [r'(?m)^(?!\s*#)\s*adb\s+-s\s+(emulator-\d+|127\.0\.0\.1:\d+)\b']),
]

# 6) Gradle Managed Device (GMD) evidence in Gradle/KTS build files.
GMD_SOURCES = [
    # DSL blocks
    ('managedDevices block', [
        r'android\s*\{[^}]*testOptions\s*\{[^}]*managedDevices\s*\{',
        r'\btestOptions\s*\{\s*managedDevices\s*\{',
        r'\bmanagedDevices\s*\{'
    ]),
    ('GMD devices/groups block', [r'\bdevices\s*\{', r'\bgroups?\s*\{']),
    # Device declarations (Groovy & Kotlin DSL)
    ('ManagedVirtualDevice type', [
        r'\(\s*ManagedVirtualDevice\s*\)',
        r'create\s*<\s*ManagedVirtualDevice\s*>\s*\(',
        r'register\s*<\s*ManagedVirtualDevice\s*>\s*\('
    ]),
    # Common properties
    ('GMD device key',            [r'\bdevice\s*=\s*["\'][^"\']+["\']']),
    ('GMD systemImageSource key', [r'\bsystemImageSource\s*=\s*["\'](google|aosp|google[-_]?atd|aosp[-_]?atd|google_apis|google_apis_playstore)["\']']),
    # apiLevel near managedDevices/ManagedVirtualDevice
    ('GMD apiLevel near managedDevices', [
        r'(managedDevices\s*\{[^}]{0,2000}\bapiLevel\s*=\s*\d{2})',
        r'(ManagedVirtualDevice[^\}]{0,2000}\bapiLevel\s*=\s*\d{2})',
        r'(testOptions\s*\{[^}]{0,2000}managedDevices\s*\{[^}]{0,2000}\bapiLevel\s*=\s*\d{2})'
    ]),
]

REAL_DEVICE_PATTERNS = [(label, compile_any(pats)) for label, pats in REAL_DEVICE_SOURCES]
EMULATOR_PATTERNS    = [(label, compile_any(pats)) for label, pats in EMULATOR_SOURCES]
THIRD_PARTY_PATTERNS = [(label, compile_any(pats)) for label, pats in THIRD_PARTY_SOURCES]
ADB_GENERIC_PATTERNS = [(label, compile_any(pats)) for label, pats in ADB_GENERIC_SOURCES]
ADB_HINT_PATTERNS    = [(label, compile_any(pats)) for label, pats in ADB_HINT_SOURCES]
GMD_PATTERNS         = [(label, compile_any(pats)) for label, pats in GMD_SOURCES]

# === TEST TRIGGERS ===
# Split Gradle into Gradle_GMD vs Gradle_Other.
GMD_VARIANTS = r'(?:Debug|Release|Dev|Prod|Qa|Beta|Staging)'

TRIGGER_PATTERNS = {
    'Gradle_GMD': [
        # GMD device-specific tasks: e.g., pixel6Api34DebugAndroidTest
        ('gradle GMD device *AndroidTest',   [fr'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?(?!connected)[A-Za-z]\w*(?:api?\d+)?{GMD_VARIANTS}AndroidTest\b'.format(GMD_VARIANTS=GMD_VARIANTS)]),
        # GMD group tasks: e.g., phoneAndTabletGroupDebugAndroidTest
        ('gradle GMD group *AndroidTest',    [fr'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?[A-Za-z]\w*Group{GMD_VARIANTS}AndroidTest\b'.format(GMD_VARIANTS=GMD_VARIANTS)]),
        # Setup/cleanup/aggregate
        ('gradle managedDeviceCheck',        [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*\bmanageddevicecheck\b']),
        ('gradle managedDeviceCleanUp',      [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*\bmanageddevicecleanup\b']),
        ('gradle allDevicesCheck',           [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*\balldevicescheck\b']),
        ('gradle cleanManagedDevices',       [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*\bcleanmanageddevices\b']),
        ('gradle *Setup',                    [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*\b\w+Setup\b']),
        # Common GMD GPU property flag used in CI
        ('GMD GPU property flag',            [r'-Pandroid\.testoptions\.manageddevices\.emulator\.gpu\s*=\s*\w+']),
    ],
    'Gradle_Other': [
        ('gradle connectedAndroidTest',      [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected(android)?test\b']),
        ('gradle connected.*Android.*Test',  [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b']),
        ('gradle connectedCheck',            [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connectedcheck\b']),
        ('gradle createInstrCoverage',       [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*createinstrumentationtestcoveragereport\b']),
        ('gradle runInstrumentationTests',   [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*runinstrumentationtests\b']),
        ('yaml script -> gradle connected',  [r'\bscript\s*:\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*']),
        ('gradle connected (broad)',         [r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*\b']),
    ],
    'ADB': [
        ('adb am instrument',                [r'(?mi)^(?!\s*#)\s*(adb\s+shell\s+)?am\s+instrument\b']),
    ],
    'Third_Party_Lab': [
        ('gcloud firebase',                  [r'(?mi)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
        ('saucectl',                         [r'(?mi)^(?!\s*#)\s*saucectl(\s+run)?\b']),
        ('appcenter test android',           [r'(?mi)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
    ],
}
TRIGGER_PATTERNS = {k: [(label, compile_any(pats)) for label, pats in v] for k, v in TRIGGER_PATTERNS.items()}

# === UNIT TESTS ===
UNIT_TEST_TRIGGER_PATTERNS = compile_any([
    r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?test(?!.*connected)(?!.*android)\b',
    r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?check(?!.*connected)(?!.*android)\b',
    r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?jvmtest\b',
    r'(?mi)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?build(?!.*connected)(?!.*android)\b',
    r'(?mi)^(?!\s*#)\s*(npm|yarn)\s+test\b',
    r'(?mi)^(?!\s*#)\s*run:\s*test\b',
])
UNIT_TEST_CONFIG_PATTERNS = compile_any([
    r'\btestimplementation\b',
    r'\bkotlin\(["\']test["\']\)',
    r'\bandroidTestUtil\b',
    r'\buseOrchestrator\s*=\s*true\b',
    r'junit:junit',
    r'org\.junit\.jupiter',
    r'mockito|mockk\b',
    r'\brobolectric\b',
    r'com\.google\.truth:truth',
    r'org\.hamcrest',
])

# === YAML script/command extraction ===
YAML_RUN_LINE_RE = re.compile(r'(?mi)^\s*(?:-+\s*)?(run|script)\s*:\s*(?:\|\s*|\>\s*)?(.+)$')
SCRIPT_PATH_RE = re.compile(r'(?P<path>(\./|\.\\)?[\w\-/\\\.]+(\.sh|\.bat|\.cmd|\.ps1|\.py|\.js|\.mjs|\.ts)?)\b')
INTERPRETER_CALL_RE = re.compile(r'(?mi)\b(bash|sh|pwsh|powershell|cmd(\.exe)?|call|python|node|npm|yarn|ruby|gradlew(\.bat)?)\b.*?(?P<path>(\./|\.\\)?[\w\-/\\\.]+(\.sh|\.bat|\.cmd|\.ps1|\.py|\.js|\.mjs|\.ts)?)')

def find_referenced_scripts(yaml_text: str) -> Set[str]:
    refs = set()
    for m in YAML_RUN_LINE_RE.finditer(yaml_text):
        chunk = (m.group(2) or '').strip()
        if not chunk:
            continue
        for p in INTERPRETER_CALL_RE.finditer(chunk):
            refs.add(os.path.normpath(p.group('path')))
        for p in SCRIPT_PATH_RE.finditer(chunk):
            refs.add(os.path.normpath(p.group('path')))
    return refs

def in_known_ci_dir(filename: str) -> bool:
    norm = filename.replace('\\', '/').lower()
    return any(norm.startswith(d.lower() + '/') or ('/' + d.lower() + '/') in norm for d in KNOWN_CI_DIRS)

# === Disambiguation helpers ===
EMULATOR_SERIAL_RE = re.compile(r'^(emulator-\d+|127\.0\.0\.1:\d+)$', re.I)
FTL_DEVICE_RE = re.compile(r'--device\b[^\\\n]*\bmodel=(?P<model>[^, \t\r\n]+)', re.I)

def split_serials(text: str):
    found = re.findall(r'(?mi)^\s*adb\s+-s\s+(?P<serial>\S+)\b', text)
    real_like = [s for s in found if not EMULATOR_SERIAL_RE.match(s)]
    emu_like  = [s for s in found if EMULATOR_SERIAL_RE.match(s)]
    return real_like, emu_like

def infer_ftl_kind(text: str):
    m = FTL_DEVICE_RE.search(text)
    if not m:
        return None, None
    model = m.group('model')
    if re.search(r'\.(arm|x86|x86_64)\b', model, re.I):
        return 'Third_Party_Lab_Virtual', model
    return 'Third_Party_Lab_Physical_or_Unknown', model

# === Summaries ===
def summarize_device_setup(text: str):
    types, details = set(), []

    emulator_hits = collect_matches(EMULATOR_PATTERNS, text)
    lab_hits      = collect_matches(THIRD_PARTY_PATTERNS, text)
    real_hits     = collect_matches(REAL_DEVICE_PATTERNS, text)
    adb_hits      = collect_matches(ADB_GENERIC_PATTERNS, text)
    adb_hint_hits = collect_matches(ADB_HINT_PATTERNS, text)
    gmd_hits      = collect_matches(GMD_PATTERNS, text)

    # Gradle Managed Devices
    if gmd_hits:
        types.add('Gradle_Managed_Device')
        details.extend([f'Gradle_Managed_Device[{h}]' for h in gmd_hits])

    # Emulator (manual/GHA)
    if emulator_hits:
        types.add('Emulator')
        details.extend([f'Emulator[{h}]' for h in emulator_hits])

    # Third-party lab
    if lab_hits:
        types.add('Third_Party_Lab')
        details.extend([f'Third_Party_Lab[{h}]' for h in lab_hits])
        kind, model = infer_ftl_kind(text)
        if kind:
            details.append(f'{kind}[{model}]')

    # Serial-based real-vs-emu split
    real_like, emu_like = split_serials(text)

    # ADB hints → promote type
    if adb_hint_hits:
        if any('emulator' in h for h in adb_hint_hits):
            types.add('Emulator')
            details.extend([f'Emulator[{h}]' for h in adb_hint_hits if 'emulator' in h])
        if any('real' in h for h in adb_hint_hits):
            types.add('Real_Device')
            details.extend([f'Real_Device[{h}]' for h in adb_hint_hits if 'real' in h])

    # Non-emulator serials → Real device
    if real_like:
        types.add('Real_Device')
        details.append(f'Real_Device[serials:{",".join(real_like[:2])}]')

    # If we have generic ADB, promote based on context; else Unknown
    if adb_hits:
        if ('Emulator' in types or 'Gradle_Managed_Device' in types) and not real_like:
            details.extend([f'Emulator_from_ADB[{h}]' for h in adb_hits])
        elif 'Third_Party_Lab' in types:
            details.extend([f'Lab_with_ADB[{h}]' for h in adb_hits])
        elif 'Real_Device' in types:
            details.extend([f'Real_with_ADB[{h}]' for h in adb_hits])
        else:
            types.add('Device_Unknown')
            details.extend([f'Device_Unknown[{h}]' for h in adb_hits])

    # Include SaaS real-device signatures (if any matched)
    if real_hits:
        types.add('Real_Device')
        details.extend([f'Real_Device[{h}]' for h in real_hits if not h.startswith('adb -s')])

    # Precedence cleanup
    if ('Third_Party_Lab' in types or 'Emulator' in types or 'Gradle_Managed_Device' in types) and not real_like and 'Real_Device' in types:
        types.discard('Real_Device')

    return types, details

def summarize_test_triggers(text: str):
    types, details = set(), []
    for ttype, named_pats in TRIGGER_PATTERNS.items():
        hits = collect_matches(named_pats, text)
        if hits:
            types.add(ttype)
            details.extend([f"{ttype}[{h}]" for h in hits])
    return types, details

# === First pass: read & pre-process all files ===
files = []
for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue
    ext = os.path.splitext(file)[-1].lower()
    if ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml', '.bat', '.cmd', '.ps1']:
        continue
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()
    cleaned = strip_comments(raw, ext)
    files.append({'name': file, 'path': file_path, 'ext': ext, 'content': cleaned})

# === Gate discovery (YAML only) ===
instrumentation_gate = False
referenced_scripts_global: Set[str] = set()
yaml_results_temp: Dict[str, Dict] = {}

for f in files:
    if f['ext'] not in ('.yml', '.yaml'):
        continue

    file = f['name']
    content = f['content'].lower()

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    ds_types, ds_detail = summarize_device_setup(content)
    trig_types, trig_detail = summarize_test_triggers(content)

    if ds_types or trig_types:
        instrumentation_gate = True

    refs = find_referenced_scripts(content)
    referenced_scripts_global |= refs

    yaml_results_temp[file] = {
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup_types': ds_types,
        'device_setup_detail': ds_detail,
        'trigger_types': trig_types,
        'trigger_detail': trig_detail,
        'parsed_ok': True
    }

# === Second pass: produce final rows with strict linking & gating ===
results = []

for f in files:
    file = f['name']; ext = f['ext']; content = f['content'].lower()

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    parsed_ok = True
    file_is_build = ext in ['.gradle', '.kts']
    file_is_yaml = ext in ['.yml', '.yaml']

    device_setup_types, device_setup_detail = set(), []
    trigger_types, trigger_detail = set(), []
    unit_test_trigger = False
    unit_test_config = False
    instru_test_status = "None"

    try:
        if file_is_yaml:
            y = yaml_results_temp.get(file, None)
            if y:
                device_setup_types = set(y['device_setup_types'])
                device_setup_detail = list(y['device_setup_detail'])
                trigger_types = set(y['trigger_types'])
                trigger_detail = list(y['trigger_detail'])

                platform_key = (y['ci_platform'] or '').strip().lower()
                has_device_ctx = bool(device_setup_types)
                has_trigger = bool(trigger_types)

                if has_device_ctx and has_trigger:
                    instru_test_status = "Complete"
                elif has_trigger and not has_device_ctx:
                    instru_test_status = "Defective" if platform_key in STRICT_CI_PLATFORMS else "Complete"
                elif has_device_ctx and not has_trigger:
                    instru_test_status = "Manual"
                else:
                    instru_test_status = "None"

        else:
            # Always scan build files for GMD & config (no YAML gate!)
            if file_is_build:
                ds, dsdet = summarize_device_setup(content)  # picks up GMD blocks & apiLevel proximity
                device_setup_types |= ds
                device_setup_detail.extend(dsdet)
                if any_match(UNIT_TEST_CONFIG_PATTERNS, content):
                    unit_test_config = True

            # Scripts are gated to reduce false positives
            if instrumentation_gate and ext in ('.sh', '.ps1', '.bat', '.cmd', '.json'):
                normfile = os.path.normpath('./' + file)
                is_referenced = (file in referenced_scripts_global) or (normfile in referenced_scripts_global)
                allowed_by_dir = LOOSEN_KNOWN_CI_PATHS and in_known_ci_dir(file)
                if is_referenced or allowed_by_dir:
                    ds, dsdet = summarize_device_setup(content)
                    tg, tgdet = summarize_test_triggers(content)
                    device_setup_types |= ds
                    device_setup_detail.extend(dsdet)
                    trigger_types |= tg
                    trigger_detail.extend(tgdet)
                    if any_match(UNIT_TEST_TRIGGER_PATTERNS, content):
                        unit_test_trigger = True

    except Exception:
        parsed_ok = False
        device_setup_types, device_setup_detail = set(), []
        trigger_types, trigger_detail = set(), []
        unit_test_trigger = False
        unit_test_config = False
        instru_test_status = "Error while parsing"

    results.append({
        'filename': file,
        'file_type': ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup': "None" if not device_setup_types else ", ".join(sorted(device_setup_types)),
        'device_setup_detail': "; ".join(device_setup_detail) if device_setup_detail else "",
        'has_device_setup': bool(device_setup_types),
        'has_test_trigger': bool(trigger_types),
        'test_trigger': ", ".join(sorted(trigger_types)) if trigger_types else "",
        'test_trigger_detail': "; ".join(trigger_detail) if trigger_detail else "",
        'unit_test_trigger': unit_test_trigger,
        'unit_test_config': unit_test_config,
        'instru_test_status': ("Complete" if (device_setup_types and trigger_types)
                                else "Manual" if device_setup_types
                                else "None"),
        'parsed_ok': parsed_ok
    })

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete. Saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete. Saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv
